In [2]:
from langchain_huggingface import ChatHuggingFace, HuggingFaceEndpoint
from dotenv import load_dotenv
from langchain_core.prompts import PromptTemplate
from langgraph.graph import StateGraph, START, END
from langgraph.checkpoint.memory import InMemorySaver
from typing import TypedDict
import time

In [3]:
class crash_state(TypedDict):

    step1: str
    step2: str
    step3: str
    step4: str

In [11]:
def step2(state: crash_state) -> crash_state:

    print("Entered step 2!!!!")

    return {'step2' : "Step 2 completed!!!"}

def step3(state: crash_state) -> crash_state:

    print("Entered step 3!!!!")

    time.sleep(30)

    return {'step3' : "Step 3 completed!!!"}

def step4(state: crash_state) -> crash_state:

    print("Entered step 4!!!!")

    return {'step4' : "Step 4 completed!!!"}

In [ ]:
graph = StateGraph(crash_state)

graph.add_node('step2', step2)
graph.add_node('step3', step3)
graph.add_node('step4', step4)

graph.add_edge(START, 'step2')
graph.add_edge('step2', 'step3')
graph.add_edge('step3', 'step4')
graph.add_edge('step4', END)

checkpointer = InMemorySaver()

workflow = graph.compile(checkpointer=checkpointer)

In [6]:
config1 = {"configurable" : {"thread_id" : "1"}}
result = workflow.invoke({'step1':"Entered Step 1"}, config=config1)

Entered step 2!!!!
Entered step 3!!!!


KeyboardInterrupt: 

In [7]:
workflow.get_state(config1)

StateSnapshot(values={'step1': 'Entered Step 1', 'step2': 'Step 2 completed!!!'}, next=('step3',), config={'configurable': {'thread_id': '1', 'checkpoint_ns': '', 'checkpoint_id': '1f13a85e-6a10-6d9d-8001-b26942e87fbe'}}, metadata={'source': 'loop', 'step': 1, 'parents': {}}, created_at='2026-04-17T17:50:19.957179+00:00', parent_config={'configurable': {'thread_id': '1', 'checkpoint_ns': '', 'checkpoint_id': '1f13a85e-6a0f-60ef-8000-3c39f9c8bc3b'}}, tasks=(PregelTask(id='450bc725-ead1-c31d-cafb-eb9d90b2f34c', name='step3', path=('__pregel_pull', 'step3'), error=None, interrupts=(), state=None, result=None),), interrupts=())

In [8]:
list(workflow.get_state_history(config1))

[StateSnapshot(values={'step1': 'Entered Step 1', 'step2': 'Step 2 completed!!!'}, next=('step3',), config={'configurable': {'thread_id': '1', 'checkpoint_ns': '', 'checkpoint_id': '1f13a85e-6a10-6d9d-8001-b26942e87fbe'}}, metadata={'source': 'loop', 'step': 1, 'parents': {}}, created_at='2026-04-17T17:50:19.957179+00:00', parent_config={'configurable': {'thread_id': '1', 'checkpoint_ns': '', 'checkpoint_id': '1f13a85e-6a0f-60ef-8000-3c39f9c8bc3b'}}, tasks=(PregelTask(id='450bc725-ead1-c31d-cafb-eb9d90b2f34c', name='step3', path=('__pregel_pull', 'step3'), error=None, interrupts=(), state=None, result=None),), interrupts=()),
 StateSnapshot(values={'step1': 'Entered Step 1'}, next=('step2',), config={'configurable': {'thread_id': '1', 'checkpoint_ns': '', 'checkpoint_id': '1f13a85e-6a0f-60ef-8000-3c39f9c8bc3b'}}, metadata={'source': 'loop', 'step': 0, 'parents': {}}, created_at='2026-04-17T17:50:19.956445+00:00', parent_config={'configurable': {'thread_id': '1', 'checkpoint_ns': '', 'c

In [9]:
workflow.invoke(None, config=config1)

Entered step 3!!!!
Entered step 4!!!!


{'step1': 'Entered Step 1', 'step2': 'Step 4 completed!!!'}

In [10]:
list(workflow.get_state_history(config1))

[StateSnapshot(values={'step1': 'Entered Step 1', 'step2': 'Step 4 completed!!!'}, next=(), config={'configurable': {'thread_id': '1', 'checkpoint_ns': '', 'checkpoint_id': '1f13a863-2ecc-6f74-8003-0ecbaaaffb6b'}}, metadata={'source': 'loop', 'step': 3, 'parents': {}}, created_at='2026-04-17T17:52:27.960502+00:00', parent_config={'configurable': {'thread_id': '1', 'checkpoint_ns': '', 'checkpoint_id': '1f13a863-2ecb-64f8-8002-d5ec63b195f8'}}, tasks=(), interrupts=()),
 StateSnapshot(values={'step1': 'Entered Step 1', 'step2': 'Step 3 completed!!!'}, next=('step4',), config={'configurable': {'thread_id': '1', 'checkpoint_ns': '', 'checkpoint_id': '1f13a863-2ecb-64f8-8002-d5ec63b195f8'}}, metadata={'source': 'loop', 'step': 2, 'parents': {}}, created_at='2026-04-17T17:52:27.959808+00:00', parent_config={'configurable': {'thread_id': '1', 'checkpoint_ns': '', 'checkpoint_id': '1f13a85e-6a10-6d9d-8001-b26942e87fbe'}}, tasks=(PregelTask(id='2cc1ec66-3217-3593-7970-a47a4dcb2881', name='step4